In [2]:
import os
import numpy as np
import cv2
import nibabel as nib

from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score

2026-03-23 18:43:31.196185: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774291411.395884      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774291411.454783      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774291411.875020      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774291411.875056      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774291411.875058      55 computation_placer.cc:177] computation placer alr

In [3]:
!pip install nibabel -q

In [4]:
IMG_SIZE = 224
BATCH_SIZE = 8

# IMPORTANT: adjust if needed
base_path = "/kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas"

image_path = os.path.join(base_path, "imagesTr")
mask_path = os.path.join(base_path, "labelsTr")

# Get patient files
patient_files = [f for f in os.listdir(image_path) if f.endswith(".nii") and not f.startswith("._")]

print("Total patients:", len(patient_files))

Total patients: 281


In [5]:
train_files, test_files = train_test_split(patient_files, test_size=0.2, random_state=42)

print("Train patients:", len(train_files))
print("Test patients:", len(test_files))

Train patients: 224
Test patients: 57


In [6]:
class CTGenerator(Sequence):
    def __init__(self, patient_files, batch_size=BATCH_SIZE, img_size=IMG_SIZE):
        self.patient_files = patient_files
        self.batch_size = batch_size
        self.img_size = img_size
        self.samples = self._prepare_samples()

    def _prepare_samples(self):
        samples = []

        for file in self.patient_files:
            img_file = os.path.join(image_path, file)
            mask_file = os.path.join(mask_path, file)

            nii_img = nib.load(img_file)
            nii_mask = nib.load(mask_file)

            volume = nii_img.get_fdata()
            mask = nii_mask.get_fdata()

            for i in range(volume.shape[2]):
                slice_mask = mask[:, :, i]

                # Label logic
                if np.any(slice_mask == 2):
                    label = 1  # tumor
                else:
                    label = 0  # normal

                samples.append((img_file, i, label))

        np.random.shuffle(samples)
        return samples

    def __len__(self):
        return int(np.ceil(len(self.samples) / self.batch_size))

    def __getitem__(self, idx):
        batch = self.samples[idx*self.batch_size:(idx+1)*self.batch_size]

        X, y = [], []

        for file, slice_idx, label in batch:
            nii = nib.load(file)
            volume = nii.get_fdata()

            slice_img = volume[:, :, slice_idx]

            # Normalize
            slice_img = (slice_img - np.min(slice_img)) / (np.max(slice_img) + 1e-8)
            slice_img = (slice_img * 255).astype(np.uint8)

            slice_img = cv2.resize(slice_img, (self.img_size, self.img_size))
            slice_img = cv2.cvtColor(slice_img, cv2.COLOR_GRAY2RGB)

            X.append(slice_img)
            y.append(label)

        return np.array(X)/255.0, np.array(y)

In [7]:
train_gen = CTGenerator(train_files)
test_gen = CTGenerator(test_files)

In [8]:
input_tensor = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

base_model = MobileNetV2(weights='imagenet', include_top=False, input_tensor=input_tensor)

x = base_model.output
x = GlobalAveragePooling2D()(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/tmp/ipykernel_55/3569897157.py:3: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_tensor=input_tensor)
I0000 00:00:1774291930.537895      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 2,225,153 (8.49 MB)

 Non-trainable params: 34,112 (133.25 KB)

In [ ]:
import numpy as np
import nibabel as nib
import os

tumor_count = 0
normal_count = 0

for file in train_files[:20]:  # check first 20 patients
    mask = nib.load(os.path.join(mask_path, file)).get_fdata()

    for i in range(mask.shape[2]):
        slice_mask = mask[:, :, i]

        if np.any(slice_mask == 2):
            tumor_count += 1
        else:
            normal_count += 1

print("Tumor slices:", tumor_count)
print("Normal slices:", normal_count)
print("Tumor ratio:", tumor_count / (tumor_count + normal_count))

In [ ]:
y_true, y_pred = [], []

for X_batch, y_batch in test_gen:
    preds = model.predict(X_batch)
    preds_label = (preds > 0.5).astype(int)

    y_true.extend(y_batch)
    y_pred.extend(preds_label.flatten())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1-score:", f1_score(y_true, y_pred))
print("ROC-AUC:", roc_auc_score(y_true, y_pred))

In [ ]:
def predict_patient(model, file):
    nii_img = nib.load(os.path.join(image_path, file))
    volume = nii_img.get_fdata()

    slices = []

    for i in range(volume.shape[2]):
        slice_img = volume[:, :, i]

        slice_img = (slice_img - np.min(slice_img)) / (np.max(slice_img)+1e-8)
        slice_img = (slice_img * 255).astype(np.uint8)

        slice_img = cv2.resize(slice_img, (IMG_SIZE, IMG_SIZE))
        slice_img = cv2.cvtColor(slice_img, cv2.COLOR_GRAY2RGB)

        slices.append(slice_img)

    slices = np.array(slices)/255.0

    preds = model.predict(slices)

    # Average prediction
    return int(preds.mean() > 0.5)

In [ ]:
y_true, y_pred = [], []

for file in test_files:
    # Ground truth from mask
    mask = nib.load(os.path.join(mask_path, file)).get_fdata()

    if np.any(mask == 2):
        true_label = 1
    else:
        true_label = 0

    pred = predict_patient(model, file)

    y_true.append(true_label)
    y_pred.append(pred)

print("Patient Accuracy:", accuracy_score(y_true, y_pred))
print("Patient Recall:", recall_score(y_true, y_pred))
print("Patient F1:", f1_score(y_true, y_pred))
print("Patient ROC-AUC:", roc_auc_score(y_true, y_pred))